<a href="https://colab.research.google.com/github/NikosMav/DataAnalysis-Netflix/blob/main/netflix_dense_retrieval.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Dense + hybrid catalog retrieval (walkthrough)

**Author:** Nikolaos (Nikos) Mavrapidis ([NikosMav](https://github.com/NikosMav))

Companion walkthrough for the `retrieval` package. Full case study, metrics table, and limitations live in [`RETRIEVAL.md`](RETRIEVAL.md). For clone-and-run use:

```bash
python -m retrieval query "war between vietnam and usa" --method hybrid
python -m retrieval eval --failures
```

This notebook is **not** a replacement for [`netflix_data_analysis.ipynb`](netflix_data_analysis.ipynb) (EDA + Boolean/TF-IDF case study).

In [ ]:
from pathlib import Path
import sys
from IPython.display import display

ROOT = Path.cwd()
if not (ROOT / "retrieval").exists() and (ROOT.parent / "retrieval").exists():
    ROOT = ROOT.parent
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))

from retrieval import BooleanRetriever, DenseRetriever, HybridRetriever, SparseTfidfRetriever, load_catalog
from retrieval.evaluate import run_evaluation, results_to_markdown

catalog = load_catalog(ROOT / "data" / "netflix_titles.csv")
len(catalog), catalog.columns.tolist()[:8]

## Build retrievers

Boolean (binary BoW + Jaccard), TF-IDF + cosine, dense MiniLM, and hybrid RRF (TF-IDF + dense).

In [ ]:
boolean = BooleanRetriever(catalog)
tfidf = SparseTfidfRetriever(catalog)
dense = DenseRetriever(catalog, text_field="text", show_progress=True)
hybrid = HybridRetriever(catalog, retrievers=[tfidf, dense], name="hybrid(tfidf+dense)")
dense.embeddings.shape

## Same demo queries as the sparse notebook

In [ ]:
def side_by_side(query, top_k=8):
    frames = {
        "boolean": boolean.query(query, top_k)[["rank", "score", "title"]],
        "tf-idf": tfidf.query(query, top_k)[["rank", "score", "title"]],
        "dense": dense.query(query, top_k)[["rank", "score", "title"]],
        "hybrid": hybrid.query(query, top_k)[["rank", "score", "title"]],
    }
    for name, frame in frames.items():
        print("=" * 20, name, "=" * 20)
        display(frame)

for q in ["war between vietnam and usa", "Mickey Mouse"]:
    print("#" * 72)
    print("QUERY:", q)
    side_by_side(q)

## Reproduce the metrics table

Uses [`data/labeled_queries.json`](data/labeled_queries.json) (28 hand-labeled queries). Same code path as `python -m retrieval eval`.

In [ ]:
metrics = run_evaluation(show_progress=False)
display(metrics)
print(results_to_markdown(metrics))

## Limitations

Catalog search ≠ production recommender ≠ TESSI ≠ RAG-over-the-web. See [`RETRIEVAL.md`](RETRIEVAL.md).